In [3]:
from collections import OrderedDict

MONTHS = []
month_to_idx = OrderedDict()

k = 0
for year in range(2020, 2026):
    for month in range(1, 13):
        MONTHS.append((year, month))
        month_to_idx[(year, month)] = k
        k += 1

T = len(MONTHS)
print("T =", T)
print(MONTHS[:3], "...", MONTHS[-3:])

T = 72
[(2020, 1), (2020, 2), (2020, 3)] ... [(2025, 10), (2025, 11), (2025, 12)]


In [4]:
import numpy as np
import rasterio
from pathlib import Path

DATA_ROOT = Path("/shared-docker/data/makeathon-challenge")
S2_ROOT = DATA_ROOT / "sentinel-2"
S1_ROOT = DATA_ROOT / "sentinel-1"
AEF_PCA_ROOT = Path("/shared-docker/cache/aligned_aef_pca")

def get_reference_s2_file(tile_id, split):
    s2_dir = S2_ROOT / split / f"{tile_id}__s2_l2a"
    files = sorted(s2_dir.glob("*.tif"))
    if not files:
        raise FileNotFoundError(f"No S2 files for {tile_id} in {split}")
    return files[0]

def get_hw(tile_id, split):
    ref = get_reference_s2_file(tile_id, split)
    with rasterio.open(ref) as src:
        return src.height, src.width

tile_id = "18NWG_6_6"
split = "train"
H, W = get_hw(tile_id, split)

s2_seq = np.zeros((T, H, W, 5), dtype=np.float32)  # B04, B08, B11, B12, s2_valid
s1_seq = np.zeros((T, H, W, 3), dtype=np.float32)  # vv_db, s1_valid, both_orbits
print("S2 seq shape:", s2_seq.shape)
print("S1 seq shape:", s1_seq.shape)

S2 seq shape: (72, 1002, 1002, 5)
S1 seq shape: (72, 1002, 1002, 3)


In [5]:
import re

S2_BAND_TO_INDEX = {
    "B04": 4,
    "B08": 8,
    "B11": 11,
    "B12": 12,
}

def parse_year_month_from_s2_name(path):
    # 18NWG_6_6__s2_l2a_2020_1.tif
    m = re.search(r"_(\d{4})_(\d{1,2})\.tif$", path.name)
    return int(m.group(1)), int(m.group(2))

def fill_s2_sequence_basic(tile_id, split, s2_seq):
    s2_dir = S2_ROOT / split / f"{tile_id}__s2_l2a"
    files = sorted(s2_dir.glob("*.tif"))
    
    for f in files:
        year, month = parse_year_month_from_s2_name(f)
        if (year, month) not in month_to_idx:
            continue
        t = month_to_idx[(year, month)]

        with rasterio.open(f) as src:
            b04 = src.read(S2_BAND_TO_INDEX["B04"]).astype(np.float32)
            b08 = src.read(S2_BAND_TO_INDEX["B08"]).astype(np.float32)
            b11 = src.read(S2_BAND_TO_INDEX["B11"]).astype(np.float32)
            b12 = src.read(S2_BAND_TO_INDEX["B12"]).astype(np.float32)

        valid = (
            np.isfinite(b04) & np.isfinite(b08) &
            np.isfinite(b11) & np.isfinite(b12) &
            (b04 > 0) & (b08 > 0) & (b11 > 0) & (b12 > 0)
        )

        s2_seq[t, :, :, 0] = np.where(valid, b04, 0.0)
        s2_seq[t, :, :, 1] = np.where(valid, b08, 0.0)
        s2_seq[t, :, :, 2] = np.where(valid, b11, 0.0)
        s2_seq[t, :, :, 3] = np.where(valid, b12, 0.0)
        s2_seq[t, :, :, 4] = valid.astype(np.float32)

    return s2_seq

s2_seq = fill_s2_sequence_basic(tile_id, split, s2_seq)
print("S2 filled months:", int((s2_seq[..., 4].sum(axis=(1,2)) > 0).sum()))
print("Example valid fraction month 0:", float(s2_seq[0, :, :, 4].mean()))

S2 filled months: 72
Example valid fraction month 0: 0.9999980330467224


In [7]:
def power_to_db(x):
    out = np.zeros_like(x, dtype=np.float32)
    valid = np.isfinite(x) & (x > 0)
    out[valid] = 10.0 * np.log10(x[valid])
    return out, valid

def parse_year_month_orbit_from_s1_name(path):
    # 18NWG_6_6__s1_rtc_2020_10_ascending.tif
    m = re.search(r"_(\d{4})_(\d{1,2})_(ascending|descending)\.tif$", path.name)
    return int(m.group(1)), int(m.group(2)), m.group(3)

def fill_s1_sequence_basic(tile_id, split, s1_seq):
    s1_dir = S1_ROOT / split / f"{tile_id}__s1_rtc"
    files = sorted(s1_dir.glob("*.tif"))

    month_groups = {}
    for f in files:
        year, month, orbit = parse_year_month_orbit_from_s1_name(f)
        if (year, month) not in month_to_idx:
            continue
        month_groups.setdefault((year, month), {})[orbit] = f

    for (year, month), d in month_groups.items():
        t = month_to_idx[(year, month)]
        arrs = []

        for orbit in ["ascending", "descending"]:
            if orbit in d:
                with rasterio.open(d[orbit]) as src:
                    x = src.read(1).astype(np.float32)
                arrs.append(x)

        if not arrs:
            continue

        mean_power = np.mean(arrs, axis=0)
        vv_db, valid = power_to_db(mean_power)
        both_orbits = 1.0 if ("ascending" in d and "descending" in d) else 0.0

        s1_seq[t, :, :, 0] = np.where(valid, vv_db, 0.0)
        s1_seq[t, :, :, 1] = valid.astype(np.float32)
        s1_seq[t, :, :, 2] = both_orbits

    return s1_seq

s1_seq = fill_s1_sequence_basic(tile_id, split, s1_seq)
print("S1 filled months:", int((s1_seq[..., 1].sum(axis=(1,2)) > 0).sum()))
print("Example both-orbits month 0:", float(s1_seq[0, :, :, 2].max()))

ValueError: could not broadcast input array from shape (334,335) into shape (1002,1002)

In [8]:
from pathlib import Path
from collections import OrderedDict
import re
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm


# =========================
# CONFIG
# =========================
DATA_ROOT = Path("/shared-docker/data/makeathon-challenge")
S2_ROOT = DATA_ROOT / "sentinel-2"
S1_ROOT = DATA_ROOT / "sentinel-1"

# AEF PCA rasters already produced by you
AEF_PCA_ROOT = Path("/shared-docker/cache/aligned_aef_pca")

# Optional cloud masks root. Set to None if you want to ignore cloud masks for now.
CLOUD_MASK_ROOT = Path("/shared-docker/cache/cloud_masks")

# Output tile packs
PACK_ROOT = Path("/shared-docker/cache/tcn_tile_packs")
PACK_ROOT.mkdir(parents=True, exist_ok=True)

# Sentinel-2 packaged band mapping (B10 absent)
S2_BAND_TO_INDEX = {
    "B04": 4,
    "B08": 8,
    "B11": 11,
    "B12": 12,
}


# =========================
# GLOBAL MONTH INDEX
# =========================
MONTHS = []
month_to_idx = OrderedDict()
k = 0
for year in range(2020, 2026):
    for month in range(1, 13):
        MONTHS.append((year, month))
        month_to_idx[(year, month)] = k
        k += 1

T = len(MONTHS)  # 72


# =========================
# BASIC HELPERS
# =========================
def get_reference_s2_file(tile_id: str, split: str) -> Path:
    s2_dir = S2_ROOT / split / f"{tile_id}__s2_l2a"
    files = sorted(s2_dir.glob("*.tif"))
    if not files:
        raise FileNotFoundError(f"No S2 files found in {s2_dir}")
    return files[0]


def get_hw(tile_id: str, split: str):
    ref = get_reference_s2_file(tile_id, split)
    with rasterio.open(ref) as src:
        return src.height, src.width


def parse_year_month_from_s2_name(path: Path):
    m = re.search(r"_(\d{4})_(\d{1,2})\.tif$", path.name)
    if m is None:
        raise ValueError(f"Could not parse S2 year/month from {path.name}")
    return int(m.group(1)), int(m.group(2))


def parse_year_month_orbit_from_s1_name(path: Path):
    m = re.search(r"_(\d{4})_(\d{1,2})_(ascending|descending)\.tif$", path.name)
    if m is None:
        raise ValueError(f"Could not parse S1 year/month/orbit from {path.name}")
    return int(m.group(1)), int(m.group(2)), m.group(3)


def power_to_db(x: np.ndarray):
    out = np.zeros_like(x, dtype=np.float32)
    valid = np.isfinite(x) & (x > 0)
    out[valid] = 10.0 * np.log10(x[valid])
    return out, valid


def read_s2_band(src, band_name: str):
    return src.read(S2_BAND_TO_INDEX[band_name]).astype(np.float32)


def read_s1_on_s2_grid(s1_path: Path, ref_s2_path: Path):
    with rasterio.open(ref_s2_path) as ref:
        dst_shape = (ref.height, ref.width)
        dst_transform = ref.transform
        dst_crs = ref.crs

    with rasterio.open(s1_path) as src:
        dst = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )

    return dst


# =========================
# CLOUD MASK LOADING
# =========================
def find_cloud_mask_path(tile_id: str, year: int, month: int, split: str):
    """
    Edit this if your cloud-mask filenames differ.
    Expected example:
      /shared-docker/cache/cloud_masks/train/18NWG_6_6_2020_1_cloudmask.tif
    """
    if CLOUD_MASK_ROOT is None:
        return None

    candidate = CLOUD_MASK_ROOT / split / f"{tile_id}_{year}_{month}_cloudmask.tif"
    return candidate if candidate.exists() else None


def load_cloud_mask(tile_id: str, year: int, month: int, split: str, ref_shape):
    """
    Returns a boolean mask where True means pixel is VALID.
    If no cloud mask exists, returns all-True.
    """
    mask_path = find_cloud_mask_path(tile_id, year, month, split)
    if mask_path is None:
        return np.ones(ref_shape, dtype=bool)

    with rasterio.open(mask_path) as src:
        m = src.read(1)

    # Assumes cloudmask is 1 for cloud/bad and 0 for clear/good.
    # If your encoding is reversed, flip this line.
    valid = (m == 0)
    return valid


# =========================
# S2 SEQUENCE
# =========================
def build_empty_s2_seq(H, W):
    # channels: B04, B08, B11, B12, valid
    return np.zeros((T, H, W, 5), dtype=np.float32)


def fill_s2_sequence_basic(tile_id: str, split: str, s2_seq: np.ndarray):
    s2_dir = S2_ROOT / split / f"{tile_id}__s2_l2a"
    files = sorted(s2_dir.glob("*.tif"))

    for f in files:
        year, month = parse_year_month_from_s2_name(f)
        if (year, month) not in month_to_idx:
            continue
        t = month_to_idx[(year, month)]

        with rasterio.open(f) as src:
            b04 = read_s2_band(src, "B04")
            b08 = read_s2_band(src, "B08")
            b11 = read_s2_band(src, "B11")
            b12 = read_s2_band(src, "B12")
            ref_shape = src.shape

        base_valid = (
            np.isfinite(b04) & np.isfinite(b08) &
            np.isfinite(b11) & np.isfinite(b12) &
            (b04 > 0) & (b08 > 0) & (b11 > 0) & (b12 > 0)
        )

        cloud_valid = load_cloud_mask(tile_id, year, month, split, ref_shape)
        valid = base_valid & cloud_valid

        s2_seq[t, :, :, 0] = np.where(valid, b04, 0.0)
        s2_seq[t, :, :, 1] = np.where(valid, b08, 0.0)
        s2_seq[t, :, :, 2] = np.where(valid, b11, 0.0)
        s2_seq[t, :, :, 3] = np.where(valid, b12, 0.0)
        s2_seq[t, :, :, 4] = valid.astype(np.float32)

    return s2_seq


# =========================
# S1 SEQUENCE
# =========================
def build_empty_s1_seq(H, W):
    # channels: vv_db, valid, both_orbits
    return np.zeros((T, H, W, 3), dtype=np.float32)


def fill_s1_sequence_basic(tile_id: str, split: str, s1_seq: np.ndarray):
    s1_dir = S1_ROOT / split / f"{tile_id}__s1_rtc"
    ref_s2_path = get_reference_s2_file(tile_id, split)
    files = sorted(s1_dir.glob("*.tif"))

    month_groups = {}
    for f in files:
        year, month, orbit = parse_year_month_orbit_from_s1_name(f)
        if (year, month) not in month_to_idx:
            continue
        month_groups.setdefault((year, month), {})[orbit] = f

    for (year, month), d in month_groups.items():
        t = month_to_idx[(year, month)]
        arrs = []

        for orbit in ["ascending", "descending"]:
            if orbit in d:
                x = read_s1_on_s2_grid(d[orbit], ref_s2_path)
                arrs.append(x)

        if not arrs:
            continue

        mean_power = np.nanmean(arrs, axis=0).astype(np.float32)
        vv_db, valid = power_to_db(mean_power)
        both_orbits = 1.0 if ("ascending" in d and "descending" in d) else 0.0

        s1_seq[t, :, :, 0] = np.where(valid, vv_db, 0.0)
        s1_seq[t, :, :, 1] = valid.astype(np.float32)
        s1_seq[t, :, :, 2] = both_orbits

    return s1_seq


# =========================
# AEF PCA
# =========================
def load_aef_basic(tile_id: str, split: str, year: int = 2020):
    aef_path = AEF_PCA_ROOT / split / f"{tile_id}_{year}_aef_pca8_on_s2_grid.tif"
    if not aef_path.exists():
        raise FileNotFoundError(f"Missing AEF PCA file: {aef_path}")

    with rasterio.open(aef_path) as src:
        x = src.read().astype(np.float32)  # [C,H,W]

    x = np.nan_to_num(x, nan=0.0)
    x = np.transpose(x, (1, 2, 0))  # [H,W,C]
    return x


# =========================
# SAVE / BUILD
# =========================
def save_tile_pack(tile_id: str, split: str, s2_seq: np.ndarray, s1_seq: np.ndarray, aef: np.ndarray):
    out_path = PACK_ROOT / f"{split}_{tile_id}.npz"
    np.savez_compressed(
        out_path,
        s2_seq=s2_seq.astype(np.float32),
        s1_seq=s1_seq.astype(np.float32),
        aef=aef.astype(np.float32),
    )
    return out_path


def build_tile_pack(tile_id: str, split: str):
    H, W = get_hw(tile_id, split)

    s2_seq = build_empty_s2_seq(H, W)
    s1_seq = build_empty_s1_seq(H, W)

    s2_seq = fill_s2_sequence_basic(tile_id, split, s2_seq)
    s1_seq = fill_s1_sequence_basic(tile_id, split, s1_seq)
    aef = load_aef_basic(tile_id, split, year=2020)

    if aef.shape[:2] != (H, W):
        raise ValueError(f"AEF shape {aef.shape[:2]} does not match S2 shape {(H, W)} for {tile_id}")

    out_path = save_tile_pack(tile_id, split, s2_seq, s1_seq, aef)

    print(f"\nBuilt tile pack: {tile_id} ({split})")
    print("  s2_seq shape:", s2_seq.shape)
    print("  s1_seq shape:", s1_seq.shape)
    print("  aef shape   :", aef.shape)
    print("  S2 filled months:", int((s2_seq[..., 4].sum(axis=(1, 2)) > 0).sum()))
    print("  S1 filled months:", int((s1_seq[..., 1].sum(axis=(1, 2)) > 0).sum()))
    print("  S2 valid overall:", float(s2_seq[..., 4].mean()))
    print("  S1 valid overall:", float(s1_seq[..., 1].mean()))
    print("  Saved to:", out_path)

    return out_path


def list_tile_ids(split: str):
    s2_dir = S2_ROOT / split
    tile_ids = []
    for p in sorted(s2_dir.iterdir()):
        if p.is_dir() and p.name.endswith("__s2_l2a"):
            tile_ids.append(p.name.replace("__s2_l2a", ""))
    return tile_ids


def batch_build_tile_packs(split: str):
    tile_ids = list_tile_ids(split)
    results = {"total": len(tile_ids), "processed": 0, "failed": 0, "failures": []}

    for tile_id in tqdm(tile_ids, desc=f"Build packs {split}", unit="tile"):
        try:
            build_tile_pack(tile_id, split)
            results["processed"] += 1
        except Exception as e:
            results["failed"] += 1
            results["failures"].append((tile_id, str(e)))

    print(f"\n{split.upper()} summary")
    print(f"total    : {results['total']}")
    print(f"processed: {results['processed']}")
    print(f"failed   : {results['failed']}")

    if results["failures"]:
        print("\nFirst failures:")
        for tile_id, err in results["failures"][:5]:
            print(tile_id)
            print("  ", err)

    return results

In [9]:
tile_id = "18NWG_6_6"
split = "train"
out_path = build_tile_pack(tile_id, split)


Built tile pack: 18NWG_6_6 (train)
  s2_seq shape: (72, 1002, 1002, 5)
  s1_seq shape: (72, 1002, 1002, 3)
  aef shape   : (1002, 1002, 8)
  S2 filled months: 72
  S1 filled months: 70
  S2 valid overall: 0.9999991059303284
  S1 valid overall: 0.9722222089767456
  Saved to: /shared-docker/cache/tcn_tile_packs/train_18NWG_6_6.npz


In [10]:
loaded = np.load(out_path)
s2_valid = loaded["s2_seq"][..., 4]  # [T,H,W]

print("Per-month valid fraction, first 12 months:")
for t in range(12):
    print(t, float(s2_valid[t].mean()))

Per-month valid fraction, first 12 months:
0 0.9999980330467224
1 0.9999980330467224
2 0.9999980330467224
3 0.9999980330467224
4 0.9999980330467224
5 0.9999980330467224
6 0.9999980330467224
7 0.9999980330467224
8 0.9999980330467224
9 0.9999980330467224
10 0.9999980330467224
11 0.9999980330467224


In [11]:
year, month = 2020, 1
mask_path = find_cloud_mask_path(tile_id, year, month, split)
print("mask_path:", mask_path)
print("exists:", None if mask_path is None else mask_path.exists())

mask_path: None
exists: None


In [12]:
from pathlib import Path

cm_dir = CLOUD_MASK_ROOT / split
print("Cloud mask dir:", cm_dir)

files = sorted(cm_dir.glob("*"))
print("Num files:", len(files))
for f in files[:20]:
    print(f.name)

Cloud mask dir: /shared-docker/cache/cloud_masks/train
Num files: 0


In [19]:
from pathlib import Path
from collections import OrderedDict
import re
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm

# =========================
# PATHS
# =========================
DATA_ROOT = Path("/shared-docker/data/makeathon-challenge")
S2_ROOT = DATA_ROOT / "sentinel-2"
S1_ROOT = DATA_ROOT / "sentinel-1"

PROCESSED_ROOT = DATA_ROOT / "processed"
S2_FEATURE_ROOT = PROCESSED_ROOT / "s2_cloudmasked_features"
AEF_PCA_ROOT = PROCESSED_ROOT / "aligned_aef_pca"
PACK_ROOT = PROCESSED_ROOT / "tcn_tile_packs"
PACK_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# GLOBAL MONTH INDEX
# =========================
MONTHS = []
month_to_idx = OrderedDict()
k = 0
for year in range(2020, 2026):
    for month in range(1, 13):
        MONTHS.append((year, month))
        month_to_idx[(year, month)] = k
        k += 1
T = len(MONTHS)  # 72


# =========================
# HELPERS
# =========================
def get_reference_s2_file(tile_id: str, split: str) -> Path:
    s2_dir = S2_ROOT / split / f"{tile_id}__s2_l2a"
    files = sorted(s2_dir.glob("*.tif"))
    if not files:
        raise FileNotFoundError(f"No S2 files found in {s2_dir}")
    return files[0]

def get_hw(tile_id: str, split: str):
    ref = get_reference_s2_file(tile_id, split)
    with rasterio.open(ref) as src:
        return src.height, src.width

def parse_year_month_from_s2_name(path: Path):
    m = re.search(r"_(\d{4})_(\d{1,2})\.tif$", path.name)
    if m is None:
        raise ValueError(f"Could not parse S2 year/month from {path.name}")
    return int(m.group(1)), int(m.group(2))

def parse_year_month_orbit_from_s1_name(path: Path):
    m = re.search(r"_(\d{4})_(\d{1,2})_(ascending|descending)\.tif$", path.name)
    if m is None:
        raise ValueError(f"Could not parse S1 year/month/orbit from {path.name}")
    return int(m.group(1)), int(m.group(2)), m.group(3)

def power_to_db(x: np.ndarray):
    out = np.zeros_like(x, dtype=np.float32)
    valid = np.isfinite(x) & (x > 0)
    out[valid] = 10.0 * np.log10(x[valid])
    return out, valid

def read_s1_on_s2_grid(s1_path: Path, ref_s2_path: Path):
    with rasterio.open(ref_s2_path) as ref:
        dst_shape = (ref.height, ref.width)
        dst_transform = ref.transform
        dst_crs = ref.crs

    with rasterio.open(s1_path) as src:
        dst = np.full(dst_shape, np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return dst


# =========================
# S2 CACHED FEATURE FILES
# =========================
def find_s2_feature_path(tile_id: str, year: int, month: int, split: str):
    """
    Expected notebook output:
    /processed/s2_cloudmasked_features/<split>/<tile_dir>/<s2_stem>_features.npz

    Example:
    /shared-docker/data/makeathon-challenge/processed/s2_cloudmasked_features/train/
      18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1_features.npz
    """
    tile_dir = S2_FEATURE_ROOT / split / f"{tile_id}__s2_l2a"
    if not tile_dir.exists():
        return None

    candidates = [
        tile_dir / f"{tile_id}__s2_l2a_{year}_{month}_features.npz",
        tile_dir / f"{tile_id}__s2_l2a_{year}_{month:02d}_features.npz",
    ]
    for c in candidates:
        if c.exists():
            return c

    fuzzy = sorted(tile_dir.glob(f"{tile_id}__s2_l2a_{year}_{month}*_features.npz"))
    if fuzzy:
        return fuzzy[0]

    fuzzy = sorted(tile_dir.glob(f"{tile_id}__s2_l2a_{year}_{month:02d}*_features.npz"))
    if fuzzy:
        return fuzzy[0]

    return None


# =========================
# S2
# =========================
def build_empty_s2_seq(H, W):
    # ndvi, ndmi, nbr, b8, b11, b12, s2_clear
    return np.zeros((T, H, W, 7), dtype=np.float32)

def fill_s2_sequence_from_cached_features(tile_id: str, split: str, s2_seq: np.ndarray):
    H, W = s2_seq.shape[1], s2_seq.shape[2]

    for (year, month), t in month_to_idx.items():
        feat_path = find_s2_feature_path(tile_id, year, month, split)
        if feat_path is None:
            continue

        data = np.load(feat_path)

        ndvi = data["ndvi"].astype(np.float32)
        ndmi = data["ndmi"].astype(np.float32)
        nbr = data["nbr"].astype(np.float32)
        b8 = data["b8"].astype(np.float32)
        b11 = data["b11"].astype(np.float32)
        b12 = data["b12"].astype(np.float32)
        s2_clear = data["s2_clear"].astype(np.float32)

        if ndvi.shape != (H, W):
            raise ValueError(f"S2 cached features shape {ndvi.shape} != {(H, W)} for {feat_path}")

        s2_seq[t, :, :, 0] = np.nan_to_num(ndvi, nan=0.0)
        s2_seq[t, :, :, 1] = np.nan_to_num(ndmi, nan=0.0)
        s2_seq[t, :, :, 2] = np.nan_to_num(nbr,  nan=0.0)
        s2_seq[t, :, :, 3] = np.nan_to_num(b8,   nan=0.0)
        s2_seq[t, :, :, 4] = np.nan_to_num(b11,  nan=0.0)
        s2_seq[t, :, :, 5] = np.nan_to_num(b12,  nan=0.0)
        s2_seq[t, :, :, 6] = s2_clear

    return s2_seq


# =========================
# S1
# =========================
def build_empty_s1_seq(H, W):
    # vv_db, s1_valid, both_orbits
    return np.zeros((T, H, W, 3), dtype=np.float32)

def fill_s1_sequence_basic(tile_id: str, split: str, s1_seq: np.ndarray):
    s1_dir = S1_ROOT / split / f"{tile_id}__s1_rtc"
    ref_s2_path = get_reference_s2_file(tile_id, split)
    files = sorted(s1_dir.glob("*.tif"))

    month_groups = {}
    for f in files:
        year, month, orbit = parse_year_month_orbit_from_s1_name(f)
        if (year, month) not in month_to_idx:
            continue
        month_groups.setdefault((year, month), {})[orbit] = f

    for (year, month), d in month_groups.items():
        t = month_to_idx[(year, month)]
        arrs = []

        for orbit in ["ascending", "descending"]:
            if orbit in d:
                x = read_s1_on_s2_grid(d[orbit], ref_s2_path)
                arrs.append(x)

        if not arrs:
            continue

        mean_power = np.nanmean(arrs, axis=0).astype(np.float32)
        vv_db, valid = power_to_db(mean_power)
        both_orbits = 1.0 if ("ascending" in d and "descending" in d) else 0.0

        s1_seq[t, :, :, 0] = np.where(valid, vv_db, 0.0)
        s1_seq[t, :, :, 1] = valid.astype(np.float32)
        s1_seq[t, :, :, 2] = both_orbits

    return s1_seq


# =========================
# AEF PCA
# =========================
def load_aef_basic(tile_id: str, split: str, year: int = 2020):
    aef_path = AEF_PCA_ROOT / split / f"{tile_id}_{year}_aef_pca8_on_s2_grid.tif"
    if not aef_path.exists():
        raise FileNotFoundError(f"Missing AEF PCA file: {aef_path}")

    with rasterio.open(aef_path) as src:
        x = src.read().astype(np.float32)  # [C, H, W]

    x = np.nan_to_num(x, nan=0.0)
    x = np.transpose(x, (1, 2, 0))  # [H, W, C]
    return x


# =========================
# BUILD / SAVE
# =========================
def save_tile_pack(tile_id: str, split: str, s2_seq: np.ndarray, s1_seq: np.ndarray, aef: np.ndarray):
    out_path = PACK_ROOT / f"{split}_{tile_id}.npz"
    np.savez_compressed(
        out_path,
        s2_seq=s2_seq.astype(np.float32),
        s1_seq=s1_seq.astype(np.float32),
        aef=aef.astype(np.float32),
    )
    return out_path

def build_tile_pack(tile_id: str, split: str):
    H, W = get_hw(tile_id, split)

    s2_seq = build_empty_s2_seq(H, W)
    s1_seq = build_empty_s1_seq(H, W)

    s2_seq = fill_s2_sequence_from_cached_features(tile_id, split, s2_seq)
    s1_seq = fill_s1_sequence_basic(tile_id, split, s1_seq)
    aef = load_aef_basic(tile_id, split, year=2020)

    if aef.shape[:2] != (H, W):
        raise ValueError(f"AEF shape {aef.shape[:2]} does not match S2 shape {(H, W)} for {tile_id}")

    out_path = save_tile_pack(tile_id, split, s2_seq, s1_seq, aef)

    print(f"\nBuilt tile pack: {tile_id} ({split})")
    print("  s2_seq shape:", s2_seq.shape)
    print("  s1_seq shape:", s1_seq.shape)
    print("  aef shape   :", aef.shape)
    print("  S2 filled months:", int((s2_seq[..., 6].sum(axis=(1, 2)) > 0).sum()))
    print("  S1 filled months:", int((s1_seq[..., 1].sum(axis=(1, 2)) > 0).sum()))
    print("  S2 valid overall:", float(s2_seq[..., 6].mean()))
    print("  S1 valid overall:", float(s1_seq[..., 1].mean()))
    print("  Saved to:", out_path)

    return out_path

def list_tile_ids(split: str):
    s2_dir = S2_ROOT / split
    tile_ids = []
    for p in sorted(s2_dir.iterdir()):
        if p.is_dir() and p.name.endswith("__s2_l2a"):
            tile_ids.append(p.name.replace("__s2_l2a", ""))
    return tile_ids

def batch_build_tile_packs(split: str):
    tile_ids = list_tile_ids(split)
    results = {"total": len(tile_ids), "processed": 0, "failed": 0, "failures": []}

    for tile_id in tqdm(tile_ids, desc=f"Build packs {split}", unit="tile"):
        try:
            build_tile_pack(tile_id, split)
            results["processed"] += 1
        except Exception as e:
            results["failed"] += 1
            results["failures"].append((tile_id, str(e)))

    print(f"\n{split.upper()} summary")
    print(f"total    : {results['total']}")
    print(f"processed: {results['processed']}")
    print(f"failed   : {results['failed']}")

    if results["failures"]:
        print("\nFirst failures:")
        for tile_id, err in results["failures"][:5]:
            print(tile_id)
            print("  ", err)

    return results

In [20]:
tile_id = "18NWG_6_6"
split = "train"
out_path = build_tile_pack(tile_id, split)


Built tile pack: 18NWG_6_6 (train)
  s2_seq shape: (72, 1002, 1002, 7)
  s1_seq shape: (72, 1002, 1002, 3)
  aef shape   : (1002, 1002, 8)
  S2 filled months: 72
  S1 filled months: 70
  S2 valid overall: 0.6431242823600769
  S1 valid overall: 0.9722222089767456
  Saved to: /shared-docker/data/makeathon-challenge/processed/tcn_tile_packs/train_18NWG_6_6.npz
